# InternVideo2-6B frozen vision compatibility and memory pilot

Load the 5.94B-parameter vision-only inference path in FP16 across available T4s.
Use fixed unlabelled clips from both halves. Verify standard-attention equivalence
on a tiny fixture, then finite, normalized, repeatable real features and memory.
No actual possession labels are read and no weights are fine-tuned.


In [ ]:
from pathlib import Path
import json,os,subprocess,sys,torch
assert torch.cuda.is_available()
print('GPU:',torch.cuda.get_device_name(0),'count',torch.cuda.device_count(),flush=True)
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working/video_pilot')
WORK.mkdir(parents=True,exist_ok=True)
roots=[p.parent for p in INPUT.rglob('VISION_EXPERIMENT.md') if (p.parent/'visual_possession/internvideo.py').is_file()]
assert len(roots)==1
CODE=roots[0]
videos=list(INPUT.rglob('BAR-ATH_tactical.mp4'));assert len(videos)==1
VIDEO=videos[0]
DEPS=Path('/tmp/barca_video_python')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
               '--target',str(DEPS),'-r',str(CODE/'requirements-temporal.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
               '--target',str(DEPS),'--no-deps','-r',str(CODE/'requirements-foundation.txt')],check=True)
env=dict(os.environ,PYTHONPATH=str(DEPS)+os.pathsep+str(CODE),PYTHONDONTWRITEBYTECODE='1',
         HF_HOME='/tmp/barca_video_hf',HF_HUB_DISABLE_XET='1',HF_HUB_DISABLE_TELEMETRY='1',
         HF_HUB_DOWNLOAD_TIMEOUT='90',HF_HUB_ETAG_TIMEOUT='30',HF_HUB_DISABLE_PROGRESS_BARS='1',
         CUBLAS_WORKSPACE_CONFIG=':4096:8')
env.pop('HF_HUB_OFFLINE',None)
def run(*args):subprocess.run([sys.executable,*map(str,args)],cwd=CODE,env=env,check=True)
run('-m','unittest','discover','-s','tests','-p','test_visual_possession.py','-v')
run('-m','visual_possession','pilot','--video',VIDEO,'--model-cache','/tmp/barca_video_models','--out',WORK)
print('PRIVATE VIDEO PILOT NOTEBOOK COMPLETE',flush=True)
